# Poopidex — Train YOLO11m on AnimalClue feces dataset (all 102 species)

Trains a medium YOLO model on the [AnimalClue feces dataset](https://huggingface.co/datasets/risashinoda/feces_yolo) (ICCV 2025). Replaces the pre-trained nano model that's currently deployed.

**Estimated time:** ~3–5 hours on free Colab T4 GPU.

**Checkpoints save to Google Drive every 10 epochs** — if Colab disconnects, re-run cells 1–6 then run the **RESUME** cell instead of the fresh-start training cell.

## Setup checklist before you run
1. Runtime → Change runtime type → **T4 GPU** (free) or **A100** (Pro)
2. Have your Hugging Face token ready (https://huggingface.co/settings/tokens — Read scope is enough)
3. Make sure you have ≥10 GB free on Google Drive (for dataset + checkpoints)


## 1. Install dependencies

In [ ]:
!pip install -q --upgrade ultralytics huggingface_hub pyyaml
import torch, ultralytics
print('ultralytics', ultralytics.__version__)
print('torch', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('⚠  No GPU detected. Switch runtime to T4 or A100 before continuing.')

## 2. Authenticate to Hugging Face

In [ ]:
from huggingface_hub import login, whoami
from getpass import getpass

token = getpass('Paste your HuggingFace token (input hidden): ')
login(token=token, add_to_git_credential=False)
print('✓ logged in as:', whoami()['name'])

## 3. Mount Google Drive (for checkpoint persistence)
Checkpoints save every 10 epochs to `MyDrive/poopidex-yolo/`. If Colab disconnects mid-training, resume from there.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/poopidex-yolo'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('✓ checkpoints will save to:', DRIVE_ROOT)

## 4. Download the dataset (~3.75 GB)
Stored under `/content/feces_yolo/`. Takes ~5–10 min on Colab's fast network.

In [ ]:
from huggingface_hub import snapshot_download

DATA_DIR = '/content/feces_yolo'
print('Downloading risashinoda/feces_yolo... (this takes a few minutes)')
snapshot_download(
    repo_id='risashinoda/feces_yolo',
    repo_type='dataset',
    local_dir=DATA_DIR,
)
print('✓ download complete')
!du -sh {DATA_DIR}

## 5. Inspect dataset structure

In [ ]:
import os

for root, dirs, files in os.walk(DATA_DIR):
    depth = root.replace(DATA_DIR, '').count(os.sep)
    if depth > 3:
        continue
    indent = '  ' * depth
    print(f'{indent}{os.path.basename(root) or DATA_DIR}/')
    if files and depth <= 2:
        sample = files[:3]
        for f in sample:
            print(f'{indent}  {f}')
        if len(files) > 3:
            print(f'{indent}  ... +{len(files) - 3} more')

print('\n--- looking for splits ---')
for split in ['train', 'val', 'valid', 'test']:
    for sub in ['', 'species/', 'species']:
        p = os.path.join(DATA_DIR, sub, split, 'images')
        if os.path.isdir(p):
            count = len(os.listdir(p))
            print(f'  ✓ found {split:6s} at {p:80s} ({count} images)')

print('\n--- looking for existing data.yaml ---')
for root, dirs, files in os.walk(DATA_DIR):
    for f in files:
        if f.endswith('.yaml') or f.endswith('.yml'):
            print(f'  found: {os.path.join(root, f)}')

## 6. Build/locate `data.yaml`
If the dataset ships with one, we update its paths. Otherwise we build one from the species CSV.

In [ ]:
import os, glob, yaml, csv

DATASET_ROOT = os.path.join(DATA_DIR, 'species')  # adjust if structure differs

# Auto-detect split paths
candidates = {'train': None, 'val': None, 'test': None}
for split_name in candidates:
    for try_name in [split_name, 'valid' if split_name == 'val' else split_name]:
        p = os.path.join(DATASET_ROOT, try_name, 'images')
        if os.path.isdir(p):
            candidates[split_name] = p
            break

print('Detected splits:')
for k, v in candidates.items():
    print(f'  {k:6s} → {v}')

# Load species names
csv_path = os.path.join(DATA_DIR, 'info_feces.csv')
species_names = []
if os.path.isfile(csv_path):
    with open(csv_path) as f:
        reader = csv.DictReader(f)
        species_names = [row['Species'] for row in reader]
    print(f'\n✓ loaded {len(species_names)} species from info_feces.csv')
else:
    print('\n⚠ info_feces.csv not found — will infer from label files (slower)')
    # Fallback: read class indices from sample label files
    label_files = glob.glob(os.path.join(DATASET_ROOT, '**/labels/*.txt'), recursive=True)
    class_ids = set()
    for lf in label_files[:1000]:
        with open(lf) as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    class_ids.add(int(parts[0]))
    species_names = [f'class_{i}' for i in sorted(class_ids)]
    print(f'  found {len(species_names)} unique class IDs in labels')

# Build data.yaml
data_yaml = {
    'path': DATASET_ROOT,
    'train': os.path.relpath(candidates['train'], DATASET_ROOT) if candidates['train'] else 'train/images',
    'val':   os.path.relpath(candidates['val'],   DATASET_ROOT) if candidates['val'] else 'val/images',
    'test':  os.path.relpath(candidates['test'],  DATASET_ROOT) if candidates['test'] else None,
    'nc':    len(species_names),
    'names': species_names,
}
DATA_YAML_PATH = '/content/poopidex_data.yaml'
with open(DATA_YAML_PATH, 'w') as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)
print(f'\n✓ wrote {DATA_YAML_PATH}')
print(f'  classes: {data_yaml["nc"]}')
print(f'  train: {data_yaml["train"]}')
print(f'  val:   {data_yaml["val"]}')
print(f'  test:  {data_yaml["test"]}')

## 7. Train YOLO11m (fresh start)

**Skip this cell if resuming a disconnected session** — use the RESUME cell below instead.

Saves checkpoints to Google Drive every 10 epochs. Total time: **~3–5 hours** on T4.

In [ ]:
from ultralytics import YOLO

# Fresh-start training. Edit hyperparameters here if you want to tweak.
model = YOLO('yolo11m.pt')  # auto-downloads COCO-pretrained medium weights

results = model.train(
    data=DATA_YAML_PATH,
    epochs=100,
    imgsz=512,
    batch=8,                # T4 fits batch=8 comfortably for YOLO11m at 512px. Bump to 16 if A100.
    device=0,
    project=DRIVE_ROOT + '/runs',
    name='train',
    exist_ok=False,         # set True if you want to overwrite previous run
    save_period=10,         # save checkpoint every N epochs (for resume-on-disconnect)
    patience=30,            # early-stop if no improvement for N epochs
    workers=4,
    seed=42,
    cos_lr=True,            # cosine learning-rate schedule, slightly better than step
    optimizer='auto',
    plots=True,
)
print('\n✓ training complete')
print(f'Best weights: {results.save_dir}/weights/best.pt')
print(f'Last weights: {results.save_dir}/weights/last.pt')

## 7b. RESUME from disconnect (only run this if training was interrupted)

Picks up from `last.pt` on Google Drive and continues to epoch 100.

In [ ]:
from ultralytics import YOLO

RESUME_FROM = DRIVE_ROOT + '/runs/train/weights/last.pt'
model = YOLO(RESUME_FROM)
results = model.train(resume=True)
print('\n✓ training complete')

## 8. Validate on the held-out test set

In [ ]:
from ultralytics import YOLO

BEST_PT = DRIVE_ROOT + '/runs/train/weights/best.pt'
model = YOLO(BEST_PT)

# Run on test set if it exists, else val
split = 'test' if data_yaml.get('test') else 'val'
metrics = model.val(data=DATA_YAML_PATH, split=split, imgsz=512, device=0)
print(f'\n--- Metrics on {split} set ---')
print(f'mAP@0.5      : {metrics.box.map50:.4f}')
print(f'mAP@0.5:0.95 : {metrics.box.map:.4f}')
print(f'precision    : {metrics.box.mp:.4f}')
print(f'recall       : {metrics.box.mr:.4f}')

## 9. Download the trained model

`best.pt` is the file you want — it's the epoch with the highest val mAP.
Drop it into `scripts/_hf_space/feces_yolo.pt` locally, then re-upload the HF Space.

In [ ]:
from google.colab import files
import shutil, os

BEST_PT = DRIVE_ROOT + '/runs/train/weights/best.pt'
OUTPUT_NAME = 'feces_yolo_v2_medium.pt'

# Copy + rename for download
shutil.copy(BEST_PT, OUTPUT_NAME)
size_mb = round(os.path.getsize(OUTPUT_NAME) / 1e6, 2)
print(f'✓ {OUTPUT_NAME} ({size_mb} MB) ready')
print(f'  also persisted to Drive at: {BEST_PT}')
files.download(OUTPUT_NAME)

## 10. Next steps (after download)

1. Place the downloaded `.pt` into your local repo:
   ```bash
   mv ~/Downloads/feces_yolo_v2_medium.pt /Users/jz/Documents/Poop/scripts/_hf_space/feces_yolo.pt
   ```
   (Keep the filename `feces_yolo.pt` — that's what `server.py` looks for.)
2. Re-upload to the HF Space — use the `huggingface_hub.upload_folder` call from earlier, or:
   ```python
   from huggingface_hub import HfApi
   HfApi().upload_file(
       path_or_fileobj='scripts/_hf_space/feces_yolo.pt',
       path_in_repo='feces_yolo.pt',
       repo_id='JZ0317/poopidex-yolo',
       repo_type='space',
       commit_message='Swap in YOLO11m trained on full 102-species AnimalClue',
   )
   ```
3. HF Space rebuilds automatically (~2 min). New model is live.
4. Verify: `curl https://jz0317-poopidex-yolo.hf.space/health` — `classes` field should still show 102 (or whatever your trained model has).
5. Test in the live app at https://poopidex.vercel.app — predictions should be noticeably more accurate.